<a href="https://colab.research.google.com/github/Matheus-Alex/Entrega-prof/blob/main/ProjetoIntegrador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Projeto Integrador — Análise Exploratória de Dados
### Sistema de Gestão de Certificados Acadêmicos
---
**Objetivo:** Explorar, compreender e extrair insights dos dados de um sistema de submissão
e validação de certificados de atividades complementares acadêmicas.

**Tipo de problema:** Classificação / Análise descritiva  
**Variável-alvo:** `status` (em `submissions`) — status da submissão do certificado  
**Foco:** Análise e entendimento dos dados (sem Machine Learning nesta etapa)

## 1. Importação das Bibliotecas

In [ ]:
# Bibliotecas essenciais para análise de dados
import pandas as pd       # manipulação e análise de dados tabulares
import numpy as np        # operações numéricas e estatísticas
import matplotlib.pyplot as plt  # visualizações básicas
import seaborn as sns     # visualizações estatísticas mais avançadas
import warnings

# Configurações visuais
sns.set(style='whitegrid')              # estilo de fundo com grade
plt.rcParams['figure.figsize'] = (10, 5)  # tamanho padrão dos gráficos
warnings.filterwarnings('ignore')       # suprimir avisos não críticos

print('Bibliotecas carregadas com sucesso!')

## 2. Carregamento dos Datasets

O sistema é composto por **13 tabelas** que representam diferentes entidades do domínio:
usuários, cursos, submissões, validações, arquivos, notificações e logs de auditoria.

In [ ]:
import pandas as pd
import json # Importar a biblioteca json para carregar o bundle

# Carregamento dos arquivos JSON individuais que foram extraídos
al = pd.read_json('/content/data_repaired/audit_logs.json')
ct = pd.read_json('/content/data_repaired/categories.json')
cs = pd.read_json('/content/data_repaired/courses.json')

# Carregamento do bundle MongoDB
# Este arquivo provavelmente contém os outros datasets
try:
    with open('/content/data_repaired/mongodb_collections_bundle.json', 'r') as f:
        mongodb_bundle = json.load(f)
except json.JSONDecodeError as e:
    print(f"Erro ao carregar mongodb_collections_bundle.json: {e}")
    print("O arquivo parece estar corrompido ou truncado. Tentando um reparo heurístico...")
    try:
        with open('/content/data_repaired/mongodb_collections_bundle.json', 'r', encoding='utf-8', errors='ignore') as f:
            corrupted_content = f.read()
        # Tentativa de fechar strings e objetos/arrays que podem estar truncados
        # Isso é uma heurística e pode não ser sempre preciso.
        repaired_content = corrupted_content.rstrip() # Remover espaços em branco no final
        if not repaired_content.endswith(('}', ']')):
            # Try to close an open string or object/array if it looks truncated
            if repaired_content.count('"') % 2 != 0: # If uneven number of quotes, try to close last string
                repaired_content += '"'
            repaired_content += '}' # Assume it's an object that needs closing

        # Tenta carregar o conteúdo reparado
        mongodb_bundle = json.loads(repaired_content)
        print("Reparo heurístico aplicado. O bundle do MongoDB foi carregado (potencialmente com dados incompletos/incorretos).")
    except Exception as fix_e:
        print(f"Falha no reparo heurístico: {fix_e}")
        print("Não foi possível carregar o bundle do MongoDB devido a corrupção.")
        mongodb_bundle = None # Define como None se não puder ser carregado

# As linhas abaixo são comentadas por enquanto, pois os arquivos CSV não existem
# e os demais dataframes precisam ser extraídos do mongodb_bundle.
# ca = pd.read_csv('/content/course_activity_rules.csv') # regras de atividade por curso
# cc = pd.read_csv('/content/course_coordinators.csv')  # coordenadores de curso
# nf = pd.read_csv('/content/notifications.csv')        # notificações do sistema
# rl = pd.read_csv('/content/roles.csv')                # papéis/perfis de acesso
# sf = pd.read_csv('/content/submission_files.csv')     # arquivos das submissões
# sb = pd.read_csv('/content/submissions.csv')          # submissões de certificados
# uc = pd.read_csv('/content/user_courses.csv')         # matrículas dos alunos
# ur = pd.read_csv('/content/user_roles.csv')           # papéis atribuídos a usuários
# us = pd.read_csv('/content/users.csv')                # cadastro de usuários
# vt = pd.read_csv('/content/validations.csv')          # histórico de validações

print('Arquivos JSON individuais (audit_logs, categories, courses) carregados.')
if mongodb_bundle is not None:
    print('O bundle do MongoDB foi carregado na variável `mongodb_bundle`.')
    print('Precisamos agora inspecionar `mongodb_bundle` para extrair os demais datasets.')
else:
    print('O bundle do MongoDB não pôde ser carregado devido a corrupção.')

### Exploração Inicial dos DataFrames Carregados

In [ ]:
# === audit_logs (al) ===
print('=== DataFrame: audit_logs (al) ===')
display(al.head())
al.info()


In [ ]:
# === categories (ct) ===
print('\n=== DataFrame: categories (ct) ===')
display(ct.head())
ct.info()


In [ ]:
# === courses (cs) ===
print('\n=== DataFrame: courses (cs) ===')
display(cs.head())
cs.info()


### Horas Mínimas Exigidas por Curso

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(x='minimum_required_hours', y='name', data=cs.sort_values('minimum_required_hours', ascending=False), palette='viridis')
plt.title('Horas Mínimas Exigidas por Curso', fontsize=14, fontweight='bold')
plt.xlabel('Horas Mínimas Exigidas')
plt.ylabel('Nome do Curso')
plt.tight_layout()
plt.show()

In [ ]:
import os

print('Arquivos no diretório /content:')
for f in os.listdir('/content'):
    print(f)

In [ ]:
import os

# Criar um diretório para os dados extraídos
if not os.path.exists('/content/data'):
    os.makedirs('/content/data')

# Descompactar o arquivo zip para o novo diretório
!unzip /content/projeto_integrador_v2_mongodb_json_bundle.zip -d /content/data

In [ ]:
!file /content/projeto_integrador_v2_mongodb_json_bundle.zip

In [ ]:
!zip -FF /content/projeto_integrador_v2_mongodb_json_bundle.zip --out /content/repaired_bundle.zip

In [ ]:
print('Tentando descompactar o arquivo reparado:')
!unzip /content/repaired_bundle.zip -d /content/data_repaired

Tentando descompactar o arquivo reparado:
Archive:  /content/repaired_bundle.zip
replace /content/data_repaired/audit_logs.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
print('Arquivos no diretório /content/data_repaired após descompactar o reparado:')
import os
for f in os.listdir('/content/data_repaired'):
    print(f)

In [ ]:
print('Arquivos no diretório /content/data após descompactar:')
for f in os.listdir('/content/data'):
    print(f)

## 3. Visão Geral dos Datasets

Antes de qualquer análise, precisamos entender o tamanho e a estrutura de cada tabela.

In [ ]:
# Dicionário com todos os datasets e seus nomes descritivos
datasets = {
    'audit_logs':            al,
    'categories':            ct,
    'courses':               cs
}

# Exibe linhas, colunas e total de valores nulos por dataset
print(f'{'Dataset':<25} {'Linhas':>8} {'Colunas':>8} {'Nulos Total':>12}')
print('-' * 58)
for nome, df in datasets.items():
    nulos = df.isnull().sum().sum()
    print(f'{nome:<25} {df.shape[0]:>8,} {df.shape[1]:>8} {nulos:>12,}')

### Entidades mais Ativas em `audit_logs`

In [ ]:
# Contar o número de ações por entidade
entities_actions = al['entity_name'].value_counts()

print('Entidades com mais ações registradas em audit_logs:')
display(entities_actions.head(10).to_frame())

# Opcional: Visualizar a distribuição das 10 principais entidades
plt.figure(figsize=(10, 6))
sns.barplot(x=entities_actions.head(10).values, y=entities_actions.head(10).index, palette='viridis')
plt.title('Top 10 Entidades com Mais Ações em Audit Logs', fontsize=14, fontweight='bold')
plt.xlabel('Número de Ações')
plt.ylabel('Nome da Entidade')
plt.tight_layout()
plt.show()

### Ações mais executadas em `audit_logs`

In [ ]:
# Contar o número de ações por tipo
action_counts = al['action'].value_counts()

print('Tipos de ações mais executadas em audit_logs:')
display(action_counts.to_frame())

# Opcional: Visualizar a distribuição das ações
plt.figure(figsize=(10, 6))
sns.barplot(x=action_counts.values, y=action_counts.index, palette='viridis')
plt.title('Distribuição de Ações em Audit Logs', fontsize=14, fontweight='bold')
plt.xlabel('Número de Ocorrências')
plt.ylabel('Tipo de Ação')
plt.tight_layout()
plt.show()

## 4. Análise do Dataset Principal: `submissions`

**Justificativa:** A tabela `submissions` é o coração do sistema. Ela registra cada
tentativa de um aluno de validar horas complementares por meio de um certificado.
A coluna `status` representa o resultado desse processo e é nossa **variável-alvo**.

In [ ]:
# Estrutura completa da tabela submissions
print('=== Informações gerais ===')
sb.info()

In [ ]:
# Primeiras linhas para inspecionar os dados visualmente
sb.head()

In [ ]:
# Análise de valores ausentes por coluna
# Valores nulos podem indicar campos opcionais ou dados não preenchidos
nulos_sb = sb.isnull().sum()
pct_nulos = (nulos_sb / len(sb) * 100).round(2)

resumo_nulos = pd.DataFrame({
    'Nulos': nulos_sb,
    '% Nulos': pct_nulos
}).query('Nulos > 0').sort_values('% Nulos', ascending=False)

print('Colunas com valores ausentes em submissions:')
print(resumo_nulos.to_string())

In [ ]:
# Estatísticas descritivas das colunas numéricas
# requested_hours: horas solicitadas pelo aluno
# approved_hours: horas efetivamente aprovadas pelo validador
print('=== Estatísticas das colunas numéricas ===')
sb[['requested_hours', 'approved_hours']].describe().round(2)

## 5. Variável-Alvo: `status`

A coluna `status` indica em que etapa do ciclo de validação cada submissão se encontra.
Compreender sua distribuição é fundamental para entender o comportamento do sistema.

In [ ]:
# Distribuição absoluta e percentual de cada status
contagem_status = sb['status'].value_counts()
pct_status = (contagem_status / len(sb) * 100).round(2)

df_status = pd.DataFrame({
    'Quantidade': contagem_status,
    '%': pct_status
})

print('Distribuição da variável-alvo (status):')
print(df_status.to_string())

In [ ]:
# Gráfico de barras da distribuição de status
# Justificativa: visualização direta facilita identificar desbalanceamento entre classes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot
contagem_status.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição de Status das Submissões', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Quantidade')
axes[0].tick_params(axis='x', rotation=30)

# Pizza
axes[1].pie(
    contagem_status,
    labels=contagem_status.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('Set2', len(contagem_status))
)
axes[1].set_title('Proporção por Status', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Análise das Horas Solicitadas e Aprovadas

In [ ]:
# Comparação entre horas solicitadas e aprovadas agrupadas por status
# Insight esperado: aprovadas < solicitadas em muitos casos
horas_status = sb.groupby('status')[['requested_hours', 'approved_hours']].mean().round(2)
print('Média de horas por status:')
print(horas_status.to_string())

In [ ]:
# Boxplot comparativo das horas solicitadas por status
# Justificativa: detecta outliers e diferenças de distribuição entre grupos
plt.figure(figsize=(12, 5))
sns.boxplot(data=sb, x='status', y='requested_hours', palette='Set3')
plt.title('Horas Solicitadas por Status', fontsize=13, fontweight='bold')
plt.xlabel('Status')
plt.ylabel('Horas Solicitadas')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Taxa de aprovação das horas (aprovadas / solicitadas) para submissões aprovadas
# Justificativa: entender se os validadores costumam cortar horas mesmo ao aprovar
sb_aprovadas = sb[sb['status'] == 'approved'].copy()
sb_aprovadas['taxa_aprovacao'] = sb_aprovadas['approved_hours'] / sb_aprovadas['requested_hours']

print('Taxa de aprovação de horas (aprovadas/solicitadas):')
print(sb_aprovadas['taxa_aprovacao'].describe().round(3))

plt.figure(figsize=(10, 4))
sns.histplot(sb_aprovadas['taxa_aprovacao'].dropna(), bins=30, color='teal', kde=True)
plt.title('Distribuição da Taxa de Aprovação de Horas', fontsize=13, fontweight='bold')
plt.xlabel('Taxa (aprovadas / solicitadas)')
plt.ylabel('Frequência')
plt.tight_layout()
plt.show()

## 7. Análise por Categoria de Atividade

A tabela `categories` classifica os tipos de atividades complementares.
Cruzar com `submissions` permite descobrir quais categorias têm mais submissões e aprovações.

In [ ]:
# Merge de submissions com categories para obter o nome da categoria
sb_cat = sb.merge(ct, left_on='category_id', right_on='id', suffixes=('_sb', '_ct'))

# Quantidade de submissões por categoria
sub_por_cat = sb_cat.groupby('name')['id_sb'].count().sort_values(ascending=False)

print('Top 10 categorias com mais submissões:')
print(sub_por_cat.head(10).to_string())

In [ ]:
# Gráfico horizontal das categorias mais frequentes
plt.figure(figsize=(12, 6))
sub_por_cat.head(10).sort_values().plot(kind='barh', color='coral', edgecolor='white')
plt.title('Top 10 Categorias com Mais Submissões', fontsize=13, fontweight='bold')
plt.xlabel('Quantidade de Submissões')
plt.ylabel('Categoria')
plt.tight_layout()
plt.show()

In [ ]:
# Taxa de aprovação (%) por categoria
# Justificativa: categorias com baixa aprovação podem indicar regras mal compreendidas pelos alunos
taxa_cat = sb_cat.groupby('name').apply(
    lambda x: (x['status'] == 'approved').sum() / len(x) * 100
).round(2).sort_values(ascending=False)

print('Taxa de aprovação (%) por categoria:')
print(taxa_cat.to_string())

## 8. Análise das Validações

A tabela `validations` registra o histórico de cada decisão tomada por um validador
(aprovação, rejeição, solicitação de revisão). Permite entender o fluxo de trabalho.

In [ ]:
# Distribuição dos status das validações
print('Distribuição de validation_status:')
print(vt['validation_status'].value_counts().to_string())
print()
print('Distribuição de previous_status:')
print(vt['previous_status'].value_counts().to_string())

In [ ]:
# Fluxo de transição: de qual status para qual status as submissões transitam
# Justificativa: entender o ciclo de vida de uma submissão no sistema
fluxo = vt.groupby(['previous_status', 'validation_status']).size().reset_index(name='count')
fluxo = fluxo.sort_values('count', ascending=False)

print('Fluxo de transição de status (top 15):')
print(fluxo.head(15).to_string(index=False))

In [ ]:
# Validadores mais ativos: quem faz mais validações?
# Justificativa: detectar sobrecarga de trabalho em validadores específicos
top_validadores = vt['validator_user_id'].value_counts().head(10)

plt.figure(figsize=(10, 4))
top_validadores.plot(kind='bar', color='mediumpurple', edgecolor='white')
plt.title('Top 10 Validadores Mais Ativos', fontsize=13, fontweight='bold')
plt.xlabel('ID do Validador')
plt.ylabel('Número de Validações')
plt.tight_layout()
plt.show()

## 9. Análise dos Arquivos Enviados

A tabela `submission_files` contém os arquivos (certificados em PDF, imagem, etc.)
enviados pelos alunos. A qualidade do OCR pode influenciar a aprovação.

In [ ]:
# Tipos de arquivo mais enviados
print('Tipos de arquivo mais enviados:')
print(sf['file_type'].value_counts().to_string())
print()

# Tamanho médio dos arquivos por tipo
print('Tamanho médio (KB) por tipo de arquivo:')
print(sf.groupby('file_type')['file_size_kb'].mean().round(2).sort_values(ascending=False).to_string())

In [ ]:
print('Colunas em audit_logs (al):')
display(al.columns.tolist())

### Significado das Colunas em `audit_logs` (al)

O dataframe `audit_logs` registra eventos e ações importantes dentro do sistema. Cada linha representa um evento de auditoria, com as seguintes colunas:

-   **`_id`**: O identificador único do registro de auditoria. É o ID interno gerado pelo sistema para cada log.
-   **`entity_name`**: O nome da entidade ou recurso que foi afetado pela ação (ex: `Submission`, `User`, `Course`).
-   **`entity_id`**: O identificador único da instância específica da entidade que foi modificada (ex: o ID de uma submissão específica, o ID de um usuário).
-   **`action`**: A ação que foi realizada (ex: `create`, `update`, `delete`, `login`, `logout`, `status_change`).
-   **`details`**: Um campo que pode conter informações adicionais ou detalhes específicos sobre a ação, muitas vezes em formato JSON ou string. Pode incluir, por exemplo, os valores antigos e novos de um campo alterado.
-   **`actor_id`**: O identificador único do usuário que realizou a ação. Este é o usuário que está logado e interagindo com o sistema.
-   **`actor_role`**: O papel ou perfil do usuário que realizou a ação no momento em que ela ocorreu (ex: `student`, `admin`, `validator`).
-   **`timestamp`**: O momento exato (data e hora) em que a ação foi registrada.

In [ ]:
# Distribuição da confiança do OCR
# Justificativa: confiança baixa pode indicar que o arquivo está ilegível,
# o que pode levar à rejeição do certificado
print('Estatísticas de confiança do OCR:')
print(sf['ocr_confidence'].describe().round(3))

plt.figure(figsize=(10, 4))
sns.histplot(sf['ocr_confidence'].dropna(), bins=30, color='darkorange', kde=True)
plt.title('Distribuição da Confiança do OCR', fontsize=13, fontweight='bold')
plt.xlabel('Confiança (0 a 1)')
plt.ylabel('Frequência')
plt.tight_layout()
plt.show()

In [ ]:
# Converter a coluna 'timestamp' para o formato datetime
al['timestamp'] = pd.to_datetime(al['timestamp'])

# Ordenar o dataframe por 'timestamp' em ordem decrescente e exibir os 5 mais recentes
print('Os 5 logs mais recentes em audit_logs:')
display(al.sort_values(by='timestamp', ascending=False).head())

## 10. Análise de Usuários e Cursos

In [ ]:
# Status dos usuários no sistema
print('Status dos usuários:')
print(us['status'].value_counts().to_string())
print()

# Percentual de usuários com CPF, telefone e último login preenchidos
campos = ['cpf', 'phone', 'last_login_at']
for campo in campos:
    pct = us[campo].notna().mean() * 100
    print(f'{campo}: {pct:.1f}% preenchido')

In [ ]:
# Distribuição de matrículas ativas vs inativas
print('Matrículas ativas em user_courses:')
print(uc['is_active'].value_counts().to_string())

# Alunos com mais matrículas
print('\nAlunos com mais cursos matriculados (top 10):')
print(uc.groupby('user_id').size().sort_values(ascending=False).head(10).to_string())

In [ ]:
# Submissões por curso: cruzar user_courses com submissions
# Justificativa: identificar cursos com maior volume de certificados enviados
sb_uc = sb.merge(uc, left_on='user_course_id', right_on='id', suffixes=('_sb', '_uc'))
sb_uc_cs = sb_uc.merge(cs, left_on='course_id', right_on='id', suffixes=('', '_cs'))

sub_por_curso = sb_uc_cs.groupby('name')['id_sb'].count().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 5))
sub_por_curso.sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 10 Cursos com Mais Submissões', fontsize=13, fontweight='bold')
plt.xlabel('Quantidade de Submissões')
plt.ylabel('Curso')
plt.tight_layout()
plt.show()

## 11. Análise Temporal

Investigar o comportamento das submissões ao longo do tempo pode revelar sazonalidades
(ex.: pico de envios próximo ao prazo de integralização de horas).

In [ ]:
# Converter datas para datetime
sb['submitted_at'] = pd.to_datetime(sb['submitted_at'])
sb['ano_mes'] = sb['submitted_at'].dt.to_period('M')  # agrupa por mês

# Volume de submissões por mês
submissoes_mes = sb.groupby('ano_mes').size()

plt.figure(figsize=(14, 5))
submissoes_mes.plot(kind='line', marker='o', color='teal', linewidth=2)
plt.title('Volume de Submissões por Mês', fontsize=13, fontweight='bold')
plt.xlabel('Mês')
plt.ylabel('Quantidade de Submissões')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Evolução do status ao longo do tempo
# Justificativa: verificar se a taxa de aprovação melhorou ou piorou ao longo dos meses
status_mes = sb.groupby(['ano_mes', 'status']).size().unstack(fill_value=0)

status_mes.plot(kind='area', figsize=(14, 6), alpha=0.7,
                colormap='Set2', linewidth=1)
plt.title('Evolução dos Status de Submissão por Mês', fontsize=13, fontweight='bold')
plt.xlabel('Mês')
plt.ylabel('Quantidade')
plt.xticks(rotation=45)
plt.legend(title='Status', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()